## Creating a RAG with ChromaDB 

#### Importing libs

In [1]:
import chromadb

import json
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent.parent))
from IPython.display import Markdown

from utils.config import RAG_DATABASE_PATH, CHROMA_DB_PATH, DB_NAME
from utils.embedding import GeminiEmbeddingFunction

### Creating the database with ChromaDB

In [2]:
documents = json.loads(RAG_DATABASE_PATH.read_text(encoding="utf-8"))
texts = []
metadatas = []
ids = []

for i, doc in enumerate(documents):
    texts.append(f"{doc['neighborhood']} — {doc['section']}\n\n{doc['text']}")
    metadatas.append({
        "neighborhood": doc["neighborhood"]
    })
    ids.append(str(i))

embed_fn = GeminiEmbeddingFunction()
chroma_client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH
)
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)
db.add(
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

### Testing the query framework

In [3]:
query = "Tem muitos restaurantes?"
target_neighborhood = 'Mooca'

results = db.query(
    query_texts=[query],
    n_results=3,
    where={"neighborhood": target_neighborhood}
)
context = "\n\n".join(results["documents"][0])

Markdown(context)

Mooca — Infraestrutura

*   **Infraestrutura Completa:** O bairro oferece uma infraestrutura urbana robusta, com escolas de qualidade desde o ensino básico até o superior, hospitais renomados, supermercados, academias e centros comerciais. Essa autossuficiência de serviços é um grande diferencial para a atratividade de locação e venda de imóveis.

Mooca — Visão Geral

Olá! Que ótima escolha você fez em olhar para a Mooca. É um bairro com uma história riquíssima e que tem se mostrado uma excelente aposta para quem busca investir no mercado imobiliário. Vamos detalhar o que faz da Mooca uma região tão interessante.

### Visão geral do bairro

A Mooca é um dos bairros mais tradicionais e charmosos de São Paulo, conhecido por sua forte herança italiana, que se reflete na cultura e, claro, na gastronomia local. Embora tenha um passado industrial, a região passou por uma notável transformação, migrando para um vibrante centro residencial e se consolidando como um dos bairros mais seguros da cidade. É um lugar que consegue harmonizar a tradição com a modernidade, oferecendo um estilo de vida acolhedor e com forte senso de comunidade.

Mooca — Desenvolvimento Urbano

*   **Revitalização e Novos Empreendimentos:** A região está em constante processo de revitalização, com a construção de diversos novos empreendimentos residenciais e comerciais, incluindo o Mooca Plaza Shopping. Essa transformação de um bairro fabril para um residencial moderno aumenta significativamente o valor do metro quadrado e a demanda por imóveis.